# DSC550-T301 Data Mining
## Week 9: 9.2 Exercise: Best Model Selection and Hyperparameter Tuning / Daniel Solis Toro

1. Import the dataset and libraries

In [16]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load dataset
df = pd.read_csv("Loan_Train.csv")

# Drop Loan_ID
df = df.drop("Loan_ID", axis=1)

# Drop missing rows
df = df.dropna()

2. Prepare data for modeling

In [18]:
df = df.drop('Loan_ID', axis=1)
df_clean = df.dropna()
categorical_columns = df_clean.select_dtypes(include=['object']).columns.tolist()
categorical_columns_for_dummies = [col for col in categorical_columns if col != 'Loan_Status']
df_encoded = pd.get_dummies(df_clean, columns=categorical_columns_for_dummies, drop_first=True)

3. Split the data into a training and test set

In [20]:
X = df_encoded.drop('Loan_Status', axis=1)
y = df_encoded['Loan_Status']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

4. Create a pipeline with a min-max scaler and a KNN classifier

In [22]:
knn_pipe = Pipeline([
    ("scaler", MinMaxScaler()),
    ("knn", KNeighborsClassifier())
])

knn_pipe.fit(X_train, y_train)

y_pred_knn = knn_pipe.predict(X_test)
knn_accuracy = accuracy_score(y_test, y_pred_knn)

print("Default KNN Accuracy:", knn_accuracy)

Default KNN Accuracy: 0.7083333333333334


5. Grid search for best n_neighbors value

In [24]:
param_grid_knn = {
    "knn__n_neighbors": list(range(1, 11))
}

grid_knn = GridSearchCV(
    knn_pipe,
    param_grid_knn,
    cv=5,
    scoring="accuracy"
)

grid_knn.fit(X_train, y_train)

best_knn = grid_knn.best_estimator_
best_knn_accuracy = accuracy_score(y_test, best_knn.predict(X_test))

print("Best KNN Params:", grid_knn.best_params_)
print("Best KNN Test Accuracy:", best_knn_accuracy)

Best KNN Params: {'knn__n_neighbors': 4}
Best KNN Test Accuracy: 0.6666666666666666


6. Expand grid search to include multiple algorithms

In [26]:
knn_params = {'classifier': [KNeighborsClassifier()], 'classifier__n_neighbors': [3, 4, 5, 6, 7, 8, 9, 10]}
lr_params = {'classifier': [LogisticRegression(max_iter=1000, random_state=42)], 
             'classifier__C': [0.01, 0.1, 1, 10, 100],
             'classifier__penalty': ['l1', 'l2'], 
             'classifier__solver': ['liblinear']}
rf_params = {'classifier': [RandomForestClassifier(random_state=42)],
             'classifier__n_estimators': [10, 50, 100, 200],
             'classifier__max_depth': [None, 5, 10, 20],
             'classifier__min_samples_split': [2, 5, 10],
             'classifier__min_samples_leaf': [1, 2, 4]}

pipe_expanded = Pipeline([('min_max_scaler', MinMaxScaler()), ('classifier', KNeighborsClassifier())])
grid_search_expanded = GridSearchCV(pipe_expanded, [knn_params, lr_params, rf_params], 
                                   cv=5, scoring='accuracy', verbose=1, n_jobs=-1)
grid_search_expanded.fit(X_train, y_train)

Fitting 5 folds for each of 162 candidates, totalling 810 fits


C:\Users\danie\anaconda3\Lib\site-packages\numpy\ma\core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('min_max_scaler', MinMaxScaler()),
                                       ('classifier', KNeighborsClassifier())]),
             n_jobs=-1,
             param_grid=[{'classifier': [KNeighborsClassifier()],
                          'classifier__n_neighbors': [3, 4, 5, 6, 7, 8, 9, 10]},
                         {'classifier': [LogisticRegression(max_iter=1000,
                                                            random_state=42)],
                          'classifier__C': [0.01, 0.1, 1, 10, 100],
                          'classifier__penalty': ['l1', 'l2'],
                          'classifier__solver': ['liblinear']},
                         {'classifier': [RandomForestClassifier(random_state=42)],
                          'classifier__max_depth': [None, 5, 10, 20],
                          'classifier__min_samples_leaf': [1, 2, 4],
                          'classifier__min_samples_split': [2, 5, 10],
                          'classifier__n_estimators': [10, 50, 100, 200]}],
             scoring='accuracy', verbose=1)

7. Results summary

In [28]:
print(f"Best model: {type(grid_search_expanded.best_estimator_.named_steps['classifier']).__name__}")
print(f"Best params: {grid_search_expanded.best_params_}")
print(f"Test accuracy: {accuracy_score(y_test, grid_search_expanded.predict(X_test)):.4f}")

Best model: RandomForestClassifier
Best params: {'classifier': RandomForestClassifier(random_state=42), 'classifier__max_depth': None, 'classifier__min_samples_leaf': 4, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 100}
Test accuracy: 0.8333


## CONCLUSION: 

The default KNN classifier achieved an accuracy of 0.7083 on the test set. After tuning the number of neighbors using 5-fold cross-validation, the best KNN model selected four neighbors but achieved a slightly lower accuracy of 0.6667. Expanding the grid search to include Logistic Regression and Random Forest classifiers resulted in a significant performance improvement. The best model identified was a Random Forest classifier with 100 estimators, no maximum depth restriction, and a minimum of four samples per leaf. This model achieved the highest test accuracy of 0.8333. Overall, the results demonstrate that ensemble tree-based methods are better suited to this dataset than distance-based classifiers like KNN.